In [1]:
!pip install huggingface_hub pandas pyarrow -q

In [2]:
!python build_dataset.py --download --cache_dir ./hf_cache --out routing_table.parquet

data/features/qwen06b_20k.jsonl: 100% 73.8M/73.8M [00:01<00:00, 56.7MB/s]
data/qwen3-0.6b/responses.jsonl: 100% 196M/196M [00:03<00:00, 50.1MB/s]
data/qwen3-0.6b/judge_scores.jsonl: 100% 47.8M/47.8M [00:03<00:00, 15.6MB/s]
data/ministral-8b/responses.jsonl: 100% 404M/404M [00:06<00:00, 58.2MB/s]
data/ministral-8b/judge_scores.jsonl: 100% 47.1M/47.1M [00:01<00:00, 24.2MB/s]
data/qwen3-30b-a3b/responses.jsonl: 100% 341M/341M [00:06<00:00, 50.5MB/s]
data/qwen3-30b-a3b/judge_scores.jsonl: 100% 45.7M/45.7M [00:02<00:00, 15.5MB/s]
data/qwen3-30b-a3b-instruct/responses.js(…): 100% 373M/373M [00:07<00:00, 48.3MB/s]
data/qwen3-30b-a3b-instruct/judge_scores(…): 100% 45.6M/45.6M [00:02<00:00, 21.9MB/s]
Loading features...
Loading qwen3-0.6b...
Loading ministral-8b...
Loading qwen3-30b-a3b...
Loading qwen3-30b-a3b-instruct...
Final table: 80875 rows, 42 columns
Saved to routing_table.parquet


In [3]:
import pandas as pd
df = pd.read_parquet("routing_table.parquet")
print(df.shape)
print(df.columns.tolist())
df.head(3)

(80875, 42)
['conversation_hash', 'turn_idx', 'query_len_chars', 'query_len_words', 'query_len_sentences', 'has_question_mark', 'question_mark_count', 'has_code', 'has_math', 'has_url', 'avg_word_length', 'uppercase_ratio', 'special_char_ratio', 'newline_count', 'num_prior_turns', 'total_prior_context_chars', 'avg_prior_query_len', 'avg_prior_response_len', 'is_first_turn', 'turn_position_ratio', 'has_pronoun_reference', 'has_continuation_marker', 'has_correction_marker', 'has_elaboration_request', 'self_contained_score', 'word_overlap_prev_query', 'word_overlap_prev_response', 'is_short_query', 'difficulty', 'query', 'qwen3-0.6b_response', 'qwen3-0.6b_cost', 'qwen3-0.6b_score', 'ministral-8b_response', 'ministral-8b_cost', 'ministral-8b_score', 'qwen3-30b-a3b_response', 'qwen3-30b-a3b_cost', 'qwen3-30b-a3b_score', 'qwen3-30b-a3b-instruct_response', 'qwen3-30b-a3b-instruct_cost', 'qwen3-30b-a3b-instruct_score']


,conversation_hash,turn_idx,query_len_chars,query_len_words,query_len_sentences,has_question_mark,question_mark_count,has_code,has_math,has_url,...,qwen3-0.6b_score,ministral-8b_response,ministral-8b_cost,ministral-8b_score,qwen3-30b-a3b_response,qwen3-30b-a3b_cost,qwen3-30b-a3b_score,qwen3-30b-a3b-instruct_response,qwen3-30b-a3b-instruct_cost,qwen3-30b-a3b-instruct_score
0,0002a1d969b12fe34cfcb53c7d6bb0ba,0,556,100,5,1,1,0,0,0,...,2.0,Voici le premier exercice ! Voici **10 phrases...,134,9.0,Parfait ! Commençons l’exercice.\n\nVoici la *...,73,10.0,Bien sûr ! Voici ton exercice personnalisé. \...,107,10.0
1,0002a1d969b12fe34cfcb53c7d6bb0ba,1,9,1,1,0,0,0,0,0,...,2.0,Exactement ! **Métaphore** (et plus précisémen...,85,10.0,Bravo ! ✅ \n**Figure de style : Métaphore**\n...,137,10.0,✅ **Bonne réponse !** \n**Figure de style : M...,108,10.0
2,0007b987dcea5835082f7571a5c2829f,0,65,13,1,0,0,0,0,0,...,3.0,بالتأكيد! سنبدأ مغامرة جديدة معًا. سأكتب لك أح...,139,10.0,أهلاً بك في مغامرة لا تُنسى! 🌟 \nأنا جاهز لكت...,186,10.0,بالطبع! 🌟 \nأهلاً بك في مغامرة مُصممة خصيصاً ...,240,10.0


In [4]:
import pandas as pd

MODELS = ["qwen3-0.6b", "ministral-8b", "qwen3-30b-a3b", "qwen3-30b-a3b-instruct"]

df = pd.read_parquet("routing_table.parquet")

print("=== Missing values per model score/cost ===")
for m in MODELS:
    print(m, "missing scores:", df[f"{m}_score"].isna().sum(),
          "| missing cost:", df[f"{m}_cost"].isna().sum())

print("\n=== Average quality score per model ===")
print(df[[f"{m}_score" for m in MODELS]].mean())

print("\n=== Average cost (tokens) per model ===")
print(df[[f"{m}_cost" for m in MODELS]].mean())

print("\n=== Difficulty distribution ===")
print(df["difficulty"].value_counts().sort_index())

print("\n=== Correlation: difficulty vs each model's score ===")
for m in MODELS:
    print(m, df["difficulty"].corr(df[f"{m}_score"]))

print("\n=== How often does the CHEAPEST model already get a perfect/near-perfect score? ===")
cheap = "qwen3-0.6b"
print(f"{cheap} score >= 9:", (df[f"{cheap}_score"] >= 9).mean().round(3), "of all turns")

print("\n=== How much better is the BEST model vs the CHEAPEST, on average? ===")
best_score = df[[f"{m}_score" for m in MODELS]].max(axis=1)
print("Avg best score:", best_score.mean().round(2))
print("Avg cheapest (qwen3-0.6b) score:", df[f"{cheap}_score"].mean().round(2))

print("\n=== Cost ratio: most expensive model vs cheapest ===")
print((df[[f"{m}_cost" for m in MODELS]].mean() / df[f"{cheap}_cost"].mean()).round(2))

print("\n=== Total tokens if you used the cheapest model for EVERY turn ===")
print(df[f"{cheap}_cost"].sum())
print("=== Total tokens if you used the most expensive model for EVERY turn ===")
most_exp = df[[f"{m}_cost" for m in MODELS]].mean().idxmax()
print(most_exp, "->", df[most_exp.replace('_cost','') + '_cost'].sum())
print(f"\nFor reference, your budget is 10,000 tokens per episode/stream — "
      f"so no policy can come close to answering all {len(df)} turns; "
      f"the episode design (how you sample a 'stream' of turns) matters a lot.")

=== Missing values per model score/cost ===
qwen3-0.6b missing scores: 0 | missing cost: 0
ministral-8b missing scores: 0 | missing cost: 0
qwen3-30b-a3b missing scores: 140 | missing cost: 0
qwen3-30b-a3b-instruct missing scores: 0 | missing cost: 0

=== Average quality score per model ===
qwen3-0.6b_score                5.160816
ministral-8b_score              8.803128
qwen3-30b-a3b_score             9.353638
qwen3-30b-a3b-instruct_score    9.582566
dtype: float64

=== Average cost (tokens) per model ===
qwen3-0.6b_cost                179.657162
ministral-8b_cost              419.475920
qwen3-30b-a3b_cost             376.156526
qwen3-30b-a3b-instruct_cost    419.646541
dtype: float64

=== Difficulty distribution ===
difficulty
-9        1
-8        9
-7       27
-6       41
-5       23
-4       20
-3       45
-2       97
-1      505
 0     5835
 1    12082
 2     5470
 3     5055
 4     7036
 5     8749
 6    10651
 7    17268
 8     7353
 9      608
Name: count, dtype: int64

=== Co

In [5]:
!python eda.py

=== Missing values per model score/cost ===
qwen3-0.6b missing scores: 0 | missing cost: 0
ministral-8b missing scores: 0 | missing cost: 0
qwen3-30b-a3b missing scores: 140 | missing cost: 0
qwen3-30b-a3b-instruct missing scores: 0 | missing cost: 0

=== Average quality score per model ===
qwen3-0.6b_score                5.160816
ministral-8b_score              8.803128
qwen3-30b-a3b_score             9.353638
qwen3-30b-a3b-instruct_score    9.582566
dtype: float64

=== Average cost (tokens) per model ===
qwen3-0.6b_cost                179.657162
ministral-8b_cost              419.475920
qwen3-30b-a3b_cost             376.156526
qwen3-30b-a3b-instruct_cost    419.646541
dtype: float64

=== Difficulty distribution ===
difficulty
-9        1
-8        9
-7       27
-6       41
-5       23
-4       20
-3       45
-2       97
-1      505
 0     5835
 1    12082
 2     5470
 3     5055
 4     7036
 5     8749
 6    10651
 7    17268
 8     7353
 9      608
Name: count, dtype: int64

=== Co

In [6]:
!pip install gymnasium -q
!python router_env.py routing_table.parquet

Obs shape: (28,)
Action space: Discrete(4)

Random policy episode finished after 55 steps
Total reward: 19.8
Answered: 25 | Skipped: 30
Remaining budget: 0.0


In [7]:
import pandas as pd
from router_env import RouterEnv, MODELS
import random

df = pd.read_parquet("routing_table.parquet")

def run_policy(action_fn, seed=7):
    env = RouterEnv(df, budget=10000, episode_length=60, seed=seed)
    obs, info = env.reset(seed=seed)
    total_reward, steps = 0, 0
    while True:
        action = action_fn()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1
        if terminated or truncated:
            break
    return steps, total_reward, info['answered'], info['skipped']

print("Always cheapest:", run_policy(lambda: 0))
print("Always best:", run_policy(lambda: 3))
print("Random:", run_policy(lambda: random.randint(0, 3)))

Always cheapest: (60, 30.693100000000005, 60, 0)
Always best: (60, 34.201499999999996, 37, 23)
Random: (60, 23.400100000000005, 29, 31)


In [8]:
!pip install stable-baselines3 -q
!python train_dqn.py routing_table.parquet --timesteps 200000 --eval_episodes 30

Streaming output truncated to the last 5000 lines.
|    episodes         | 2076     |
|    fps              | 319      |
|    time_elapsed     | 380      |
|    total_timesteps  | 121609   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.131    |
|    n_updates        | 30152    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 59.6     |
|    ep_rew_mean      | 32.1     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 2080     |
|    fps              | 319      |
|    time_elapsed     | 380      |
|    total_timesteps  | 121843   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.212    |
|    n_updates        | 30210    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 59.5     |
|   

In [9]:
!python evaluate_heuristic.py routing_table.parquet --dqn_model dqn_router.zip --eval_episodes 30

Loaded 80875 rows
Difficulty tertile thresholds: low<= 3.00, mid<= 6.00, else best
2026-07-07 04:29:33.610326: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.

Policy                                  Mean Reward    Std       Avg Answered   Avg Skipped 
Always cheapest (qwen3-0.6b)            27.903         3.112     56.80          3.20        
Always best (qwen3-30b-a3b-instruct)    27.518 